# 07 — Pruning Basics

> Companion to **Week 11**, Part 6 of the lecture.

## The idea

> Many of the weights in a trained network are *tiny*. Setting them to zero
> barely changes the model's predictions — but the resulting "sparse" model
> compresses much better and can sometimes run faster on supporting hardware.

This notebook trains a small MLP, then **removes 40% of the weights** and
checks how much accuracy is lost.


## Step 1 — Train the model


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256)


class PruneMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


def sparsity(layer):
    zeros = torch.sum(layer.weight == 0).item()
    total = layer.weight.nelement()
    return zeros / total


model = PruneMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

before_acc = evaluate(model)
print(f"Accuracy BEFORE pruning: {before_acc:.3f}")
print(f"Sparsity of fc1 BEFORE:   {sparsity(model.fc1):.2%}")


## Step 2 — Unstructured pruning

We start with **global unstructured L1 pruning**: PyTorch looks at all the
weights in `fc1` and `fc2` together, finds the 40% with the smallest absolute
value, and replaces them with zero. *Individual edges* are removed — the shape
of the matrix does not change.


In [ ]:
import copy

unstructured_model = copy.deepcopy(model)

parameters_to_prune = (
    (unstructured_model.fc1, "weight"),
    (unstructured_model.fc2, "weight"),
)

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.4,
)

unstructured_acc = evaluate(unstructured_model)

pd.DataFrame(
    [
        {"metric": "accuracy_before",        "value": before_acc},
        {"metric": "accuracy_after_unstruct","value": unstructured_acc},
        {"metric": "fc1_sparsity",           "value": sparsity(unstructured_model.fc1)},
        {"metric": "fc2_sparsity",           "value": sparsity(unstructured_model.fc2)},
        {"metric": "fc1_shape (unchanged)",  "value": str(tuple(unstructured_model.fc1.weight.shape))},
    ]
)


### What you should see

- ~40% of the weights in `fc1` and `fc2` are now exactly zero.
- Accuracy is nearly the same as before — usually within 1-2 percentage points.
- **The shape of the weight matrix did not change**: `(128, 64)` is still
  `(128, 64)`. That's the key property of *unstructured* pruning.

> ⚠️ **Important caveat:** unstructured pruning does **not** automatically make
> inference faster. You still multiply the input by a full dense matrix —
> most of the entries just happen to be zero. You need a sparse matrix library
> or special hardware to turn those zeros into actual speed.

## Step 3 — Structured pruning

*Structured* pruning removes whole **rows** (output neurons) or **columns**
(input connections) of a weight matrix, so the matrix is *literally* smaller
afterwards. The matmul is smaller → real speedup on any hardware.

We use `ln_structured` which ranks rows by their L2 norm and drops the 30%
smallest.


In [ ]:
structured_model = copy.deepcopy(model)

# Drop 30% of the OUTPUT channels (rows) of fc1 — dim=0 means along rows
prune.ln_structured(
    structured_model.fc1,
    name="weight",
    amount=0.3,
    n=2,           # L2 norm
    dim=0,         # row-wise = per output neuron
)

structured_acc = evaluate(structured_model)

row_norms = structured_model.fc1.weight.norm(dim=1)
dead_rows = int((row_norms == 0).sum())

pd.DataFrame(
    [
        {"metric": "accuracy_after_structured", "value": structured_acc},
        {"metric": "fc1_total_rows",            "value": structured_model.fc1.weight.shape[0]},
        {"metric": "fc1_zeroed_rows",           "value": dead_rows},
        {"metric": "fc1_sparsity",              "value": sparsity(structured_model.fc1)},
    ]
)


### What to compare

| | Unstructured (step 2) | Structured (step 3) |
|---|---|---|
| What was removed | 40% of individual weights | 30% of whole output neurons of `fc1` |
| Matrix shape after | unchanged | unchanged *tensor*, but 30% of rows are now all zero |
| Accuracy drop | very small | larger (we removed more "all at once") |
| Speedup on a normal CPU | ❌ unless sparse kernels | ✅ if you actually drop the dead rows |

Structured pruning costs more accuracy, but its speedup is real on any
hardware — you can literally rebuild `fc1` as a smaller `nn.Linear(64, 90)`
layer and save the multiplication by the dead rows entirely.

## 🧪 Your turn

1. Push unstructured pruning to the limit: `amount = 0.6`, `0.8`, `0.95`.
   Where is the "accuracy cliff"?
2. In the structured block, raise `amount` to `0.5`, then `0.7`. How quickly
   does structured pruning fall off compared to unstructured?
3. **Bonus:** rebuild `structured_model.fc1` as a smaller `nn.Linear` that
   drops the zeroed rows, and time it vs the original. Do you see a speedup?
